#### Sử dụng mô hình Text-to-Text Transfer Transformer

In [1]:
import pandas as pd
import torch
import gc
from datasets import Dataset
from transformers import (
    T5Tokenizer, 
    T5ForConditionalGeneration, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer, 
    DataCollatorForSeq2Seq
)

/home/vophilong/Documents/Deep_learning/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### SETUP VÀ TẢI DỮ LIỆU

In [3]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [4]:
print("Đang tải và xử lý dữ liệu...")
train_df = pd.read_csv('/home/vophilong/Documents/Deep_learning/DoAn_Email/dataset/dataset_split/train.csv').dropna()
val_df = pd.read_csv('/home/vophilong/Documents/Deep_learning/DoAn_Email/dataset/dataset_split/val.csv').dropna()
test_df = pd.read_csv('/home/vophilong/Documents/Deep_learning/DoAn_Email/dataset/dataset_split/test.csv').dropna()

Đang tải và xử lý dữ liệu...


In [5]:
# Chuyển pandas dataframe sang dạng Dataset của HuggingFace để xử lý cho nhanh
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

In [6]:
# Khởi tạo Tokenizer của T5
model_name = "t5-base"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

model.gradient_checkpointing_enable()# Kích hoạt gradient checkpointing để tiết kiệm VRAM khi huấn luyện

Loading weights: 100%|██████████| 257/257 [00:00<00:00, 1649.62it/s]


In [7]:


# Hàm chuyển đổi văn bản thành Input IDs (Tokenization) tôi ưu hóa độ dài input và output để phù hợp với tài nguyên máy tính


def preprocess_function(examples):
    inputs = ["reply email: " + doc for doc in examples["Input_Text"]]
    # Giảm từ 512 xuống 256
    model_inputs = tokenizer(inputs, max_length=256, truncation=True, padding="max_length")

    # Giảm từ 256 xuống 128
    labels = tokenizer(examples["Output_Text"], max_length=128, truncation=True, padding="max_length")
    
    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label] for label in labels["input_ids"]
    ]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Đang mã hóa Token (Tokenization)...")
tokenized_train = train_dataset.map(preprocess_function, batched=True, remove_columns=train_dataset.column_names)#giúp loại bỏ các cột gốc sau khi đã mã hóa, chỉ giữ lại các cột cần thiết cho mô hình
tokenized_val = val_dataset.map(preprocess_function, batched=True, remove_columns=val_dataset.column_names)
tokenized_test = test_dataset.map(preprocess_function, batched=True, remove_columns=test_dataset.column_names)

Đang mã hóa Token (Tokenization)...


Map: 100%|██████████| 1384/1384 [00:00<00:00, 3187.36 examples/s]


In [8]:
# Công cụ tự động gộp batch dữ liệu
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model_name)

#### TỐI ƯU SIÊU THAM SỐ BẰNG OPTUNA

In [ ]:
print("Bắt đầu tối ưu siêu tham số (Hyperparameter Tuning)...")

# Hàm khởi tạo mô hình mới (Optuna cần gọi hàm này mỗi lần thử nghiệm 1 bộ số mới)
def model_init():
    return T5ForConditionalGeneration.from_pretrained(model_name)

# Cấu hình Training Arguments cơ bản
training_args = Seq2SeqTrainingArguments(
    output_dir="./t5_email_tuning",# Thư mục lưu mô hình và logs
    eval_strategy="epoch",  # Đánh giá trên tập Val mỗi epoch
    save_strategy="epoch",# Lưu mô hình mỗi epoch
    logging_dir="./logs",# Thư mục lưu logs cho TensorBoard
    predict_with_generate=True,   # Cho phép sinh chữ để đánh giá
    fp16=torch.cuda.is_available(), # Tăng tốc nếu có GPU
    disable_tqdm=False, 
)

# Khởi tạo Trainer
trainer = Seq2SeqTrainer(
    model_init=model_init,
    args=training_args,# Các tham số này sẽ được Optuna điều chỉnh trong quá trình tìm kiếm
    train_dataset=tokenized_train,# Dữ liệu huấn luyện đã được mã hóa
    eval_dataset=tokenized_val,
    processing_class=tokenizer,# API mới thay cho tokenizer trong Trainer
    data_collator=data_collator,# Công cụ tự động gộp batch dữ liệu, giúp tối ưu hiệu suất khi huấn luyện
)

# Định nghĩa không gian tìm kiếm (Search Space) cho Optuna
def optuna_hp_space(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-4, log=True),
        "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [8, 16]),# Tùy vào GPU, batch size lớn sẽ nhanh hơn nhưng tốn nhiều VRAM hơn
        "num_train_epochs": trial.suggest_int("num_train_epochs", 5, 10)
    }

# Tiến hành tìm kiếm (Chỉ chạy 3 trials để demo, nếu bạn có thời gian hãy tăng n_trials lên 10-20)
print("Đang cho AI chạy thử nghiệm các tổ hợp tham số... (Quá trình này sẽ tốn thời gian)")

best_trial = trainer.hyperparameter_search(
    direction="minimize", # Tìm Validation Loss nhỏ nhất
    backend="optuna",
    hp_space=optuna_hp_space,
    n_trials=3 
)
# Hiện tiến trình chạy thử nghiệm


print(f"SIÊU THAM SỐ TỐT NHẤT TÌM ĐƯỢC: {best_trial.hyperparameters}")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Bắt đầu tối ưu siêu tham số (Hyperparameter Tuning)...


Loading weights: 100%|██████████| 257/257 [00:00<00:00, 8289.78it/s]
[I 2026-03-30 09:11:44,154] A new study created in memory with name: no-name-b3c7f263-14d3-43b6-ad78-55ec7fe0eec0


Đang cho AI chạy thử nghiệm các tổ hợp tham số... (Quá trình này sẽ tốn thời gian)


Loading weights: 100%|██████████| 257/257 [00:00<00:00, 7605.24it/s]
/home/vophilong/Documents/Deep_learning/venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[W 2026-03-30 09:16:19,764] Trial 0 failed with parameters: {'learning_rate': 0.00041669111668686943, 'per_device_train_batch_size': 8, 'num_train_epochs': 7} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/vophilong/Documents/Deep_learning/venv/lib/python3.10/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/home/vophilong/Documents/Deep_learning/venv/lib/python3.10/site-packages/transformers/integrations/integration_utils.py", line 256, in _objective
    trainer.train(resume_from_checkpoint=checkpoint, trial=trial)
  File "/home/vophilong/Documents/Deep

KeyboardInterrupt: 